# 02 · Function Arguments: Positional, Keyword, Defaults, `*args`, `**kwargs`, `/` and `*`

**Module 05 – Functions** · Estimated time: 55 min

### 🎯 What you will learn
1. Positional vs keyword arguments
2. Default parameter values, and the **mutable default** trap
3. `*args`: any number of positional arguments
4. `**kwargs`: any number of keyword arguments
5. The full parameter order
6. 🆕 Positional-only (`/`, 3.8) and keyword-only (`*`) parameters
7. Unpacking arguments when *calling*: `f(*list)`, `f(**dict)`
8. 🆕 `sentinel` defaults (3.15)

## 1. Positional and keyword arguments

In [1]:
def student(name, age):
    print(f"Name: {name}, Age: {age}")

student("Bob", 20)               # positional: matched by ORDER
student(age=20, name="Bob")      # keyword: matched by NAME (any order)
student("Bob", age=20)           # mixed: positional first

Name: Bob, Age: 20
Name: Bob, Age: 20
Name: Bob, Age: 20


In [2]:
student(name="Bob", 20)          # ❌ SyntaxError: positional argument follows keyword argument

SyntaxError: positional argument follows keyword argument (1445756650.py, line 1)

## 2. Default values

In [3]:
def greet(name="Guest", greeting="Hello"):
    return f"{greeting}, {name}!"

print(greet())
print(greet("Asha"))
print(greet("Asha", "Namaste"))
print(greet(greeting="Hi"))

Hello, Guest!
Hello, Asha!
Namaste, Asha!
Hi, Guest!


Parameters with defaults must come **after** those without:

In [4]:
def bad(a=1, b):      # ❌ SyntaxError
    pass

SyntaxError: parameter without a default follows parameter with a default (3335604157.py, line 1)

### ❌ The mutable default argument trap

Defaults are evaluated **once**, when the function is defined, not on every call. A mutable default (list, dict, set) is therefore **shared** between calls:

In [5]:
def add_item(item, basket=[]):       # ❌ BUG
    basket.append(item)
    return basket

print(add_item("apple"))
print(add_item("banana"))            # 'apple' is still there!

['apple']
['apple', 'banana']


In [6]:
def add_item(item, basket=None):     # ✅ use None as the default
    if basket is None:
        basket = []
    basket.append(item)
    return basket

print(add_item("apple"))
print(add_item("banana"))

['apple']
['banana']


## 3. `*args`: variable positional arguments (📗 + 📘)

`*args` collects any extra positional arguments into a **tuple**. (`args` is just a convention; the `*` is what matters.)

In [7]:
def myfunc(*args):
    print(type(args), args)
    return sum(args) * 0.05

print(myfunc(40, 60, 20))
print(myfunc())

<class 'tuple'> (40, 60, 20)
6.0
<class 'tuple'> ()
0.0


In [8]:
def my_function(*args):
    for arg in args:
        print(arg)

my_function(1, 2, 3, "hello")

1
2
3
hello


In [9]:
def average(*numbers):
    if not numbers:
        return 0
    return sum(numbers) / len(numbers)

print(average(1, 2, 3, 4), average(10))

2.5 10.0


## 4. `**kwargs`: variable keyword arguments

`**kwargs` collects extra keyword arguments into a **dict**.

In [10]:
def myfunc(**kwargs):
    print(type(kwargs), kwargs)
    if "fruit" in kwargs:
        print(f"My favorite fruit is {kwargs['fruit']}")
    else:
        print("I don't like fruit")

myfunc(fruit="pineapple", veggie="lettuce")
myfunc()

<class 'dict'> {'fruit': 'pineapple', 'veggie': 'lettuce'}
My favorite fruit is pineapple
<class 'dict'> {}
I don't like fruit


In [11]:
def build_profile(name, **details):
    profile = {"name": name}
    profile.update(details)
    return profile

print(build_profile("Asha", city="Pune", role="Engineer", age=28))

{'name': 'Asha', 'city': 'Pune', 'role': 'Engineer', 'age': 28}


## 5. Combining everything: the parameter order

```python
def f(pos_only, /, normal, *args, kw_only, **kwargs):
```
Order: **positional-only → normal → `*args` → keyword-only → `**kwargs`**

In [12]:
def my_function(a, b, *args, c=10, **kwargs):
    print(f"a: {a}, b: {b}, args: {args}, c: {c}, kwargs: {kwargs}")

my_function(1, 2, 3, 4, 5, c=20, name="Bob", country="USA")
my_function(1, 2)

a: 1, b: 2, args: (3, 4, 5), c: 20, kwargs: {'name': 'Bob', 'country': 'USA'}
a: 1, b: 2, args: (), c: 10, kwargs: {}


In [13]:
def myfunc(*args, **kwargs):
    if "fruit" in kwargs and args:
        print(f"I like {' and '.join(args)} and my favorite fruit is {kwargs['fruit']}")

myfunc("eggs", "spam", fruit="cherries", juice="orange")

I like eggs and spam and my favorite fruit is cherries


### `*args`/`**kwargs` in inheritance (📘 handbook)

Subclasses can accept extra arguments and pass the rest on:

In [14]:
class Animal:
    def __init__(self, name):
        self.name = name

class Dog(Animal):
    def __init__(self, name, breed, *args, **kwargs):
        super().__init__(name)
        self.breed = breed
        print(f"{name}: args={args}, kwargs={kwargs}")

dog1 = Dog("Buddy", "Golden Retriever")
dog2 = Dog("Lucy", "Labrador", 1, 2, 3, color="Black", age=5)

Buddy: args=(), kwargs={}
Lucy: args=(1, 2, 3), kwargs={'color': 'Black', 'age': 5}


### Use cases
- **Flexible APIs**: functions like `print(*objects, sep=...)` and `max(*args, key=...)`
- **Decorators**: wrap functions with any signature (Module 09)
- **Forwarding**: pass arguments through layers of calls
- **Inheritance**: accept and forward extra parameters

## 6. Keyword-only (`*`) and 🆕 positional-only (`/`) parameters

- Parameters **after** a bare `*` must be passed **by keyword**, which makes calls self-documenting.
- Parameters **before** `/` must be passed **by position** (3.8+), which lets you rename them later without breaking callers.

In [15]:
def connect(host, port, *, timeout=10, retries=3):
    return f"{host}:{port} timeout={timeout} retries={retries}"

print(connect("localhost", 8080, timeout=5))

localhost:8080 timeout=5 retries=3


In [16]:
connect("localhost", 8080, 5)      # ❌ TypeError: timeout is keyword-only

TypeError: connect() takes 2 positional arguments but 3 were given

In [17]:
def divide(a, b, /):
    return a / b

print(divide(10, 2))

5.0


In [18]:
divide(a=10, b=2)      # ❌ TypeError: positional-only

TypeError: divide() got some positional-only arguments passed as keyword arguments: 'a, b'

In [19]:
def full(a, b, /, c, d, *, e, f):
    return a, b, c, d, e, f

print(full(1, 2, 3, d=4, e=5, f=6))

(1, 2, 3, 4, 5, 6)


Many built-ins use `/`: look at `help(len)` → `len(obj, /)`.

## 7. Unpacking arguments when calling

`*` spreads a sequence into positional arguments; `**` spreads a dict into keyword arguments.

In [20]:
def describe(name, age, city):
    return f"{name}, {age}, from {city}"

info_list = ["Ravi", 30, "Delhi"]
info_dict = {"name": "Meena", "age": 25, "city": "Chennai"}
print(describe(*info_list))
print(describe(**info_dict))
print(max(*[3, 9, 2]))
coords = [(1, 2), (3, 4)]
print(*coords)

Ravi, 30, from Delhi
Meena, 25, from Chennai
9
(1, 2) (3, 4)


## 8. 🆕 Defaults with a `sentinel` (Python 3.15)

When `None` is a legitimate value, you need another way to tell "argument not given":

In [21]:
MISSING = sentinel("MISSING")

def set_value(key, value=MISSING):
    if value is MISSING:
        return f"{key}: no value given"
    return f"{key} = {value!r}"

print(set_value("a"))
print(set_value("b", None))      # None is a real value here

a: no value given
b = None


> On older Pythons the classic trick is `_MISSING = object()`.

## 📝 Summary

| Syntax | Meaning |
|---|---|
| `def f(a, b)` | positional-or-keyword |
| `def f(a=1)` | default value (never a mutable one!) |
| `def f(*args)` | extra positionals → tuple |
| `def f(**kwargs)` | extra keywords → dict |
| `def f(a, /, b)` | `a` positional-only |
| `def f(a, *, b)` | `b` keyword-only |
| `f(*seq)`, `f(**d)` | unpack when calling |

## 🏋️ Practice (📗 *args and **kwargs homework style)
1. Write `myfunc(*args)` that returns a list of only the even arguments.
2. Write `myfunc(s)` that returns the string with alternating upper/lowercase letters (`'Anthropomorphism'` → `'aNtHrOpOmOrPhIsM'`).
3. Write `make_tag(tag, text, **attrs)` that returns `<a href="x">text</a>` for `make_tag("a", "text", href="x")`.
4. Write `safe_divide(a, b, /, *, default=0)`.
5. Fix the bug: `def log(msg, history=[]): history.append(msg); return history`.

<details><summary>Show solutions</summary>

```python
def evens(*args):
    return [n for n in args if n % 2 == 0]

def alternate(s):
    return "".join(c.upper() if i % 2 else c.lower() for i, c in enumerate(s))

def make_tag(tag, text, **attrs):
    attr_str = "".join(f' {k}="{v}"' for k, v in attrs.items())
    return f"<{tag}{attr_str}>{text}</{tag}>"

def safe_divide(a, b, /, *, default=0):
    return a / b if b != 0 else default

def log(msg, history=None):
    if history is None:
        history = []
    history.append(msg)
    return history
```
</details>